# 10. PCA: Reducción de Dimensionalidad

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 75 minutos  
**Prerequisitos:** Álgebra lineal básica (vectores, matrices)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender el concepto de reducción de dimensionalidad
- Comprender eigenvalues y eigenvectors intuitivamente
- Implementar PCA desde cero usando descomposición SVD
- Calcular e interpretar varianza explicada
- Aplicar PCA para visualización y preprocessing
- Identificar cuándo usar y cuándo no usar PCA

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_digits, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades
import sys
sys.path.append('../../shared/utils')
from visualization import plot_pca_components
from datasets import load_dataset

np.random.seed(42)
print("✅ Librerías importadas")

---
## 📌 1. Motivación: La Maldición de la Dimensionalidad

### El Problema de Alta Dimensión

**Ejemplo:** Dataset con 1000 features
- Difícil de visualizar
- Lento de entrenar
- Mucha memoria
- Riesgo de overfitting

**Pregunta:** ¿Necesitamos realmente las 1000 features?

### La Intuición de PCA

**Analogía: Fotografiar un lápiz**

```
3D (Original):              2D (Proyección):
    
      ✏️                         ____
     /                          |    |
    /                           |____|
   
Visto desde cierto ángulo,    Captura toda
el lápiz se ve como 2D        la información
```

**PCA hace exactamente esto:** Encuentra los mejores "ángulos" (direcciones) para proyectar los datos.

### Ejemplo Concreto: Datos de Estudiantes

Tienes 5 features:
1. Nota en Matemáticas
2. Nota en Física
3. Nota en Química
4. Nota en Literatura
5. Nota en Historia

**Observación:** Matemáticas, Física y Química están muy correlacionadas.

**PCA descubre:**
- **PC1** = "Habilidad científica" (combina Math, Physics, Chemistry)
- **PC2** = "Habilidad humanística" (combina Literature, History)

De 5 features → 2 componentes principales que capturan ~95% de la información.

### ¿Qué es PCA?

**Principal Component Analysis** = Encontrar las direcciones de **máxima varianza**.

```
Datos originales:           Después de PCA:

  🔴                          🔴
   🔴 🔴                      🔴🔴
  🔴   🔴    →               🔴  🔴
    🔴                        🔴
      
(datos correlacionados)     (datos decorrelacionados)
                             en nuevos ejes
```

### Aplicaciones Reales

- 📊 **Visualización**: Plotear datos de alta dimensión en 2D/3D
- 🚀 **Preprocessing**: Reducir features antes de ML
- 🗜️ **Compresión**: Reducir tamaño de datos (ej: imágenes)
- 🔍 **Feature extraction**: Encontrar features latentes importantes
- 🧬 **Bioinformática**: Análisis de expresión genética
- 💰 **Finanzas**: Análisis de carteras, encontrar factores de riesgo

### Problema Real: Reconocimiento Facial

**Dataset:** Imágenes de caras 64x64 pixels
- Dimensionalidad original: 4096 features (cada pixel)
- Con PCA: ~50-100 componentes capturan la esencia de una cara
- **Eigenfaces:** Los primeros componentes principales son "caras típicas"

**Beneficio:**
- 40x menos datos
- Clasificación más rápida
- Menos overfitting

### La Pregunta Guía

> **¿Cómo podemos encontrar automáticamente las dimensiones más importantes en datos de alta dimensión?**

---
## 📊 2. Intuición Visual

In [ ]:
# Generar datos 2D correlacionados
np.random.seed(42)
mean = [0, 0]
cov = [[3, 2.5], [2.5, 3]]  # Covarianza alta → correlación
X_2d = np.random.multivariate_normal(mean, cov, 200)

# Aplicar PCA
pca_2d = PCA(n_components=2)
pca_2d.fit(X_2d)

# Obtener componentes principales (vectores)
components = pca_2d.components_
explained_var = pca_2d.explained_variance_

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Datos originales con componentes principales
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.6, s=50)

# Dibujar componentes principales
origin = pca_2d.mean_
for i, (comp, var) in enumerate(zip(components, explained_var)):
    v = comp * 3 * np.sqrt(var)  # Escalar por varianza para visualización
    axes[0].arrow(origin[0], origin[1], v[0], v[1],
                 head_width=0.3, head_length=0.3, fc=f'C{i+1}', ec=f'C{i+1}',
                 linewidth=3, label=f'PC{i+1}')

axes[0].set_xlabel('Feature 1', fontsize=12)
axes[0].set_ylabel('Feature 2', fontsize=12)
axes[0].set_title('Datos Originales con Componentes Principales', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Plot 2: Datos proyectados en espacio PC
X_pca = pca_2d.transform(X_2d)
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, s=50)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('PC1 (Primera Componente Principal)', fontsize=12)
axes[1].set_ylabel('PC2 (Segunda Componente Principal)', fontsize=12)
axes[1].set_title('Datos Proyectados en Espacio PCA', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

print("\n💡 Observa:")
print(f"   • PC1 (naranja) captura la dirección de máxima varianza")
print(f"   • PC1 explica {pca_2d.explained_variance_ratio_[0]*100:.1f}% de la varianza")
print(f"   • PC2 (verde) es perpendicular a PC1")
print(f"   • PC2 explica {pca_2d.explained_variance_ratio_[1]*100:.1f}% de la varianza")
print(f"   • Juntas explican {sum(pca_2d.explained_variance_ratio_)*100:.1f}% (toda la información)")

In [ ]:
# Reducción de dimensionalidad: 3D → 2D
# Generar datos 3D
np.random.seed(42)
n_samples = 500

# Crear datos que yacen principalmente en un plano
t = np.linspace(0, 4*np.pi, n_samples)
x = t * np.cos(t) + np.random.normal(0, 0.5, n_samples)
y = t * np.sin(t) + np.random.normal(0, 0.5, n_samples)
z = 0.5 * t + np.random.normal(0, 0.5, n_samples)

X_3d = np.column_stack([x, y, z])

# Aplicar PCA para reducir a 2D
pca_3d = PCA(n_components=2)
X_2d_projected = pca_3d.fit_transform(X_3d)

# Visualización
fig = plt.figure(figsize=(14, 6))

# Plot 1: Datos 3D originales
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], c=t, cmap='viridis', s=20)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Datos Originales (3D)', fontsize=14)

# Plot 2: Proyección 2D
ax2 = fig.add_subplot(122)
scatter = ax2.scatter(X_2d_projected[:, 0], X_2d_projected[:, 1], 
                     c=t, cmap='viridis', s=20)
ax2.set_xlabel('PC1', fontsize=12)
ax2.set_ylabel('PC2', fontsize=12)
ax2.set_title('Proyección PCA (2D)', fontsize=14)
plt.colorbar(scatter, ax=ax2, label='Tiempo')

plt.tight_layout()
plt.show()

print(f"\n📊 Varianza explicada:")
for i, var_ratio in enumerate(pca_3d.explained_variance_ratio_):
    print(f"   PC{i+1}: {var_ratio*100:.2f}%")
print(f"   Total: {sum(pca_3d.explained_variance_ratio_)*100:.2f}%")

print("\n💡 Pasamos de 3D a 2D reteniendo ~{:.1f}% de la información".format(
    sum(pca_3d.explained_variance_ratio_)*100))

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Notación

| Símbolo | Significado |
|---------|-------------|
| $\mathbf{X}$ | Matriz de datos (n × d) |
| $\bar{\mathbf{x}}$ | Vector de medias |
| $\mathbf{C}$ | Matriz de covarianza |
| $\lambda_i$ | Eigenvalue $i$ |
| $\mathbf{v}_i$ | Eigenvector $i$ |
| $k$ | Número de componentes a retener |
| $\mathbf{W}$ | Matriz de proyección (d × k) |

---

## 3.1 El Problema de Optimización

**Objetivo:** Encontrar dirección $\mathbf{w}$ que maximice la varianza de los datos proyectados.

**Varianza de proyección:**
$$
\text{Var}(\mathbf{X}\mathbf{w}) = \mathbf{w}^T \mathbf{C} \mathbf{w} \tag{1}
$$

**Restricción:** $\|\mathbf{w}\| = 1$ (vector unitario)

**Problema de optimización:**
$$
\max_{\mathbf{w}} \mathbf{w}^T \mathbf{C} \mathbf{w} \quad \text{s.t.} \quad \mathbf{w}^T \mathbf{w} = 1 \tag{2}
$$

---

## 3.2 Solución: Eigenvalues y Eigenvectors

### Matriz de Covarianza

Para datos centrados ($\bar{\mathbf{x}} = 0$):
$$
\mathbf{C} = \frac{1}{n} \mathbf{X}^T \mathbf{X} \tag{3}
$$

### Definición de Eigenvalue/Eigenvector

Un vector $\mathbf{v}$ es un **eigenvector** de $\mathbf{C}$ con **eigenvalue** $\lambda$ si:
$$
\mathbf{C} \mathbf{v} = \lambda \mathbf{v} \tag{4}
$$

**Interpretación:**
- $\mathbf{C}$ transforma $\mathbf{v}$ en una versión escalada de sí mismo
- $\lambda$ es el factor de escala
- Direcciones especiales que no cambian de dirección bajo transformación

### Teorema Fundamental de PCA

**Los componentes principales son los eigenvectors de la matriz de covarianza, ordenados por sus eigenvalues.**

$$
\lambda_1 \geq \lambda_2 \geq ... \geq \lambda_d \geq 0 \tag{5}
$$

Donde:
- $\mathbf{v}_1$ (eigenvector con $\lambda_1$ máximo) = Primera componente principal
- $\mathbf{v}_2$ (eigenvector con $\lambda_2$ segundo) = Segunda componente principal
- ...

---

## 3.3 Varianza Explicada

**Varianza total:**
$$
\sigma_{\text{total}}^2 = \sum_{i=1}^{d} \lambda_i = \text{trace}(\mathbf{C}) \tag{6}
$$

**Varianza explicada por PC $i$:**
$$
\text{Var. Explicada}_i = \frac{\lambda_i}{\sum_{j=1}^{d} \lambda_j} \tag{7}
$$

**Varianza acumulada (primeros k componentes):**
$$
\text{Var. Acumulada}_k = \frac{\sum_{i=1}^{k} \lambda_i}{\sum_{j=1}^{d} \lambda_j} \tag{8}
$$

**Regla práctica:** Retener componentes hasta alcanzar 95% de varianza explicada.

---

## 3.4 Proyección y Reconstrucción

### Proyección (reducción de dimensionalidad)

Sea $\mathbf{W} = [\mathbf{v}_1, \mathbf{v}_2, ..., \mathbf{v}_k]$ (matriz de primeros k eigenvectors).

**Proyectar datos:**
$$
\mathbf{Z} = \mathbf{X} \mathbf{W} \tag{9}
$$

Donde:
- $\mathbf{X}$: (n × d) - datos originales
- $\mathbf{W}$: (d × k) - matriz de proyección
- $\mathbf{Z}$: (n × k) - datos proyectados (reducidos)

### Reconstrucción (de baja a alta dimensión)

**Reconstruir datos:**
$$
\hat{\mathbf{X}} = \mathbf{Z} \mathbf{W}^T \tag{10}
$$

### Error de Reconstrucción

$$
\text{MSE} = \frac{1}{n} \|\mathbf{X} - \hat{\mathbf{X}}\|_F^2 = \frac{1}{n} \sum_{i=k+1}^{d} \lambda_i \tag{11}
$$

**Interpretación:** El error es la varianza descartada (eigenvalues no usados).

---

## 3.5 Singular Value Decomposition (SVD)

**Método alternativo** (más estable numéricamente):

Descomponer $\mathbf{X}$ (centrada) como:
$$
\mathbf{X} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T \tag{12}
$$

Donde:
- $\mathbf{U}$: (n × n) - vectores singulares izquierdos
- $\mathbf{\Sigma}$: (n × d) - valores singulares (diagonal)
- $\mathbf{V}$: (d × d) - vectores singulares derechos

**Conexión con eigenvalues:**
$$
\lambda_i = \frac{\sigma_i^2}{n} \tag{13}
$$

**Componentes principales:**
$$
\mathbf{W} = \mathbf{V} \tag{14}
$$

---

## 3.6 Algoritmo PCA Completo

**Input:** Matriz de datos $\mathbf{X}$ (n × d), número de componentes $k$

**Paso 1:** Centrar los datos
$$
\bar{\mathbf{x}} = \frac{1}{n} \sum_{i=1}^{n} \mathbf{x}_i \tag{15}
$$
$$
\mathbf{X}_{\text{centrado}} = \mathbf{X} - \bar{\mathbf{x}} \tag{16}
$$

**Paso 2:** Calcular matriz de covarianza
$$
\mathbf{C} = \frac{1}{n} \mathbf{X}_{\text{centrado}}^T \mathbf{X}_{\text{centrado}} \tag{17}
$$

**Paso 3:** Calcular eigenvalues y eigenvectors de $\mathbf{C}$
$$
\mathbf{C} \mathbf{v}_i = \lambda_i \mathbf{v}_i \tag{18}
$$

**Paso 4:** Ordenar eigenvectors por eigenvalue (descendente)

**Paso 5:** Seleccionar primeros $k$ eigenvectors
$$
\mathbf{W} = [\mathbf{v}_1, \mathbf{v}_2, ..., \mathbf{v}_k] \tag{19}
$$

**Paso 6:** Proyectar datos
$$
\mathbf{Z} = \mathbf{X}_{\text{centrado}} \mathbf{W} \tag{20}
$$

**Output:** Datos transformados $\mathbf{Z}$ (n × k)

### Ejemplo Numérico

**Datos:** 3 puntos en 2D

$$
\mathbf{X} = \begin{bmatrix}
1 & 2 \\
2 & 3 \\
3 & 4
\end{bmatrix}
$$

**Paso 1: Centrar**
$$
\bar{\mathbf{x}} = [2, 3]^T
$$
$$
\mathbf{X}_{\text{c}} = \begin{bmatrix}
-1 & -1 \\
0 & 0 \\
1 & 1
\end{bmatrix}
$$

**Paso 2: Covarianza**
$$
\mathbf{C} = \frac{1}{3}
\begin{bmatrix}
-1 & 0 & 1 \\
-1 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
-1 & -1 \\
0 & 0 \\
1 & 1
\end{bmatrix}
=
\begin{bmatrix}
0.67 & 0.67 \\
0.67 & 0.67
\end{bmatrix}
$$

**Paso 3: Eigenvalues**
- $\lambda_1 = 1.33$ (99.7% varianza)
- $\lambda_2 = 0.004$ (0.3% varianza)

**Interpretación:** Los datos yacen casi perfectamente en una línea 1D.

In [ ]:
# Verificar ejemplo numérico
X_example = np.array([[1, 2], [2, 3], [3, 4]])

# Centrar
X_centered = X_example - X_example.mean(axis=0)
print("Datos centrados:")
print(X_centered)

# Covarianza
C = (X_centered.T @ X_centered) / len(X_example)
print("\nMatriz de covarianza:")
print(C)

# Eigenvalues y eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(C)
print("\nEigenvalues:")
print(eigenvalues)
print("\nVarianza explicada:")
print(eigenvalues / eigenvalues.sum())

print("\n✅ El primer componente explica ~99.7% de la varianza")

---
## 💻 4. Implementación Desde Cero

In [ ]:
class PrincipalComponentAnalysis:
    """
    Implementación desde cero de PCA usando SVD.
    
    Parameters:
    -----------
    n_components : int or float, default=None
        Número de componentes a retener.
        - Si int: número exacto de componentes
        - Si float (0-1): varianza mínima a explicar
        - Si None: retener todos
    """
    
    def __init__(self, n_components=None):
        self.n_components = n_components
        
        # Se inicializan durante fit
        self.mean_ = None
        self.components_ = None  # Eigenvectors (componentes principales)
        self.explained_variance_ = None  # Eigenvalues
        self.explained_variance_ratio_ = None
        self.n_features_ = None
        self.n_samples_ = None
    
    def fit(self, X):
        """
        Calcula los componentes principales.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        
        Returns:
        --------
        self : PrincipalComponentAnalysis
        """
        self.n_samples_, self.n_features_ = X.shape
        
        print(f"🔍 Aplicando PCA a datos de shape {X.shape}\n")
        
        # Paso 1: Centrar los datos
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # Paso 2: Calcular SVD
        # X = U * S * Vt
        # U: (n × n), S: (min(n,d),), Vt: (d × d)
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        
        # Paso 3: Componentes principales = V (transpuesta de Vt)
        # Vt está ordenado por valores singulares descendentes
        self.components_ = Vt  # (d × d) o (min(n,d) × d)
        
        # Paso 4: Calcular varianza explicada
        # Eigenvalues = (valores singulares)^2 / n
        self.explained_variance_ = (S ** 2) / (self.n_samples_ - 1)
        
        # Ratio de varianza explicada
        total_var = self.explained_variance_.sum()
        self.explained_variance_ratio_ = self.explained_variance_ / total_var
        
        # Paso 5: Determinar número de componentes a retener
        if self.n_components is None:
            n_components = self.n_features_
        elif isinstance(self.n_components, float):
            # Interpretar como varianza mínima a explicar
            cumsum_var = np.cumsum(self.explained_variance_ratio_)
            n_components = np.searchsorted(cumsum_var, self.n_components) + 1
            print(f"   Reteniendo {n_components} componentes para explicar "
                  f"{self.n_components*100:.1f}% de varianza")
        else:
            n_components = min(self.n_components, self.n_features_)
        
        # Reducir a n_components
        self.components_ = self.components_[:n_components]
        self.explained_variance_ = self.explained_variance_[:n_components]
        self.explained_variance_ratio_ = self.explained_variance_ratio_[:n_components]
        
        print(f"📊 Varianza explicada por componente:")
        cumsum = 0
        for i, var_ratio in enumerate(self.explained_variance_ratio_):
            cumsum += var_ratio
            print(f"   PC{i+1}: {var_ratio*100:6.2f}%  (acum: {cumsum*100:6.2f}%)")
        
        print(f"\n✅ PCA completado: {self.n_features_} → {len(self.components_)} dimensiones")
        
        return self
    
    def transform(self, X):
        """
        Proyecta datos al espacio de componentes principales.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        
        Returns:
        --------
        X_transformed : array, shape (n_samples, n_components)
        """
        # Centrar con la media del conjunto de entrenamiento
        X_centered = X - self.mean_
        
        # Proyectar: Z = X * W
        return X_centered @ self.components_.T
    
    def fit_transform(self, X):
        """Fit y transform en un solo paso"""
        self.fit(X)
        return self.transform(X)
    
    def inverse_transform(self, X_transformed):
        """
        Reconstruye datos originales desde espacio PCA.
        
        Parameters:
        -----------
        X_transformed : array-like, shape (n_samples, n_components)
        
        Returns:
        --------
        X_reconstructed : array, shape (n_samples, n_features)
        """
        # Reconstruir: X_hat = Z * W^T + mean
        return X_transformed @ self.components_ + self.mean_

print("✅ Clase PrincipalComponentAnalysis definida")

In [ ]:
# Probar con dataset Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Normalizar (importante para PCA)
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Aplicar nuestro PCA
pca_custom = PrincipalComponentAnalysis(n_components=2)
X_pca_custom = pca_custom.fit_transform(X_iris_scaled)

In [ ]:
# Visualizar resultado
fig = go.Figure()

for i, species in enumerate(iris.target_names):
    mask = y_iris == i
    fig.add_trace(go.Scatter(
        x=X_pca_custom[mask, 0],
        y=X_pca_custom[mask, 1],
        mode='markers',
        marker=dict(size=10, opacity=0.7),
        name=species
    ))

fig.update_layout(
    title="PCA de Iris Dataset (4D → 2D)",
    xaxis_title="Primera Componente Principal (PC1)",
    yaxis_title="Segunda Componente Principal (PC2)",
    template="plotly_white",
    font=dict(size=12),
    width=700,
    height=500
)

fig.show()

print("\n💡 Las 3 especies de Iris son claramente separables en espacio PCA")

In [ ]:
# Demostración de reconstrucción
# Reducir a 2 dimensiones y luego reconstruir a 4
X_reconstructed = pca_custom.inverse_transform(X_pca_custom)

# Calcular error de reconstrucción
mse = np.mean((X_iris_scaled - X_reconstructed) ** 2)

print("🔄 Reconstrucción de Datos:\n")
print(f"Original shape: {X_iris_scaled.shape}")
print(f"PCA shape: {X_pca_custom.shape}")
print(f"Reconstructed shape: {X_reconstructed.shape}")
print(f"\nMSE de reconstrucción: {mse:.6f}")

# Mostrar ejemplo
idx = 0
print(f"\nEjemplo (muestra {idx}):")
print(f"Original:       {X_iris_scaled[idx]}")
print(f"PCA (2D):       {X_pca_custom[idx]}")
print(f"Reconstructed:  {X_reconstructed[idx]}")
print(f"Error:          {X_iris_scaled[idx] - X_reconstructed[idx]}")

print("\n💡 Perdimos información (2 componentes descartados) pero retenemos lo principal")

---
## 🏭 5. Versión con Framework (Scikit-learn)

In [ ]:
# Comparar con Scikit-learn
pca_sklearn = PCA(n_components=2)
X_pca_sklearn = pca_sklearn.fit_transform(X_iris_scaled)

print("📊 Comparación: Nuestra Implementación vs Scikit-learn\n")
print("="*70)
print(f"{'Métrica':<35} {'Nuestra':<17} {'Scikit-learn':<17}")
print("="*70)

for i in range(2):
    print(f"PC{i+1} Varianza Explicada (%)  "
          f"{pca_custom.explained_variance_ratio_[i]*100:<17.2f} "
          f"{pca_sklearn.explained_variance_ratio_[i]*100:<17.2f}")

print("="*70)
print(f"Total Varianza Explicada (%)  "
      f"{sum(pca_custom.explained_variance_ratio_)*100:<17.2f} "
      f"{sum(pca_sklearn.explained_variance_ratio_)*100:<17.2f}")
print("="*70)

print("\n✅ ¡Resultados idénticos! Nuestra implementación es correcta")

In [ ]:
# Aplicación: PCA como preprocessing para clasificación
# Cargar dataset de alta dimensión
digits = load_digits()
X_digits = digits.data  # 64 features (8x8 pixels)
y_digits = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42
)

# Normalizar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("🔢 Dataset MNIST Digits:")
print(f"   Dimensionalidad original: {X_digits.shape[1]} features")
print(f"   Samples: {len(X_train)} train, {len(X_test)} test\n")

# Experimento: Clasificar con diferentes números de componentes
n_components_list = [5, 10, 20, 30, 40, 64]
results = []

for n_comp in n_components_list:
    # PCA
    pca = PCA(n_components=n_comp)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    # Clasificar
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train_pca, y_train)
    acc = clf.score(X_test_pca, y_test)
    
    var_explained = sum(pca.explained_variance_ratio_)
    
    results.append({
        'n_components': n_comp,
        'accuracy': acc,
        'var_explained': var_explained
    })
    
    print(f"n_components={n_comp:2d}: Acc={acc:.4f}, "
          f"Var. Explicada={var_explained*100:.1f}%")

print("\n💡 Con solo 20-30 componentes (de 64) obtenemos buen accuracy")

In [ ]:
# Visualizar trade-off: dimensionalidad vs accuracy
results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=results_df['n_components'],
    y=results_df['accuracy'],
    mode='lines+markers',
    name='Test Accuracy',
    marker=dict(size=10, color='blue'),
    yaxis='y1'
))

fig.add_trace(go.Scatter(
    x=results_df['n_components'],
    y=results_df['var_explained'] * 100,
    mode='lines+markers',
    name='Varianza Explicada (%)',
    marker=dict(size=10, color='red'),
    yaxis='y2'
))

fig.update_layout(
    title="Trade-off: Dimensionalidad vs Accuracy",
    xaxis=dict(title="Número de Componentes PCA"),
    yaxis=dict(title="Test Accuracy", side='left', range=[0.8, 1.0]),
    yaxis2=dict(title="Varianza Explicada (%)", side='right', overlaying='y', range=[0, 105]),
    template="plotly_white",
    font=dict(size=12)
)

fig.show()

In [ ]:
# Visualizar los primeros componentes principales como imágenes
# (Eigenfaces para dígitos)
pca_full = PCA(n_components=16)
pca_full.fit(X_train_scaled)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
axes = axes.ravel()

for i in range(16):
    # Reshape componente principal a imagen 8x8
    component_image = pca_full.components_[i].reshape(8, 8)
    
    axes[i].imshow(component_image, cmap='RdBu_r')
    axes[i].set_title(f'PC{i+1}\n({pca_full.explained_variance_ratio_[i]*100:.1f}%)',
                     fontsize=10)
    axes[i].axis('off')

plt.suptitle('Primeros 16 Componentes Principales ("Eigendigits")', 
            fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

print("\n💡 Cada componente captura un patrón común en los dígitos")
print("   Los primeros componentes capturan estructuras globales")
print("   Los últimos capturan detalles finos")

### Cuándo Usar (y No Usar) PCA

#### ✅ Usa PCA si:

1. **Features muy correlacionadas**
   - Elimina redundancia
   - Ej: temperatura en °C y °F

2. **Visualización de alta dimensión**
   - Proyectar a 2D/3D
   - Exploración de datos

3. **Reducir overfitting**
   - Menos features = menos riesgo
   - Regularización implícita

4. **Acelerar entrenamiento**
   - Menos dimensiones = más rápido
   - Importante para datasets grandes

5. **Preprocesamiento para otros algoritmos**
   - Antes de clustering
   - Antes de SVM en alta dimensión

#### ❌ NO uses PCA si:

1. **Features tienen significado interpretable importante**
   - PCA mezcla features
   - Pierdes interpretabilidad

2. **Relaciones no-lineales**
   - PCA solo captura varianza lineal
   - Considera Kernel PCA, t-SNE, UMAP

3. **Features en diferentes escalas y NO normalizas**
   - PCA sensible a escala
   - Siempre normalizar primero

4. **Supervised learning y quieres maximizar separabilidad**
   - PCA ignora labels
   - Considera LDA (Linear Discriminant Analysis)

5. **Datos sparse (muchos ceros)**
   - PCA densifica datos
   - Pierdes eficiencia de sparsity

---
## 🎯 6. Ejercicios

### 🟢 Ejercicio 1: Elegir Número de Componentes

Implementa una función para elegir automáticamente el número de componentes.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Elegir k óptimo basado en varianza explicada
    
    Instrucciones:
    1. Usa Breast Cancer dataset
    2. Aplica PCA con todos los componentes
    3. Crea Scree Plot (varianza vs componente)
    4. Identifica "codo" donde varianza marginal < threshold
    5. Retorna número de componentes que explican 95% varianza
    
    Returns:
    --------
    k_95 : int
        Número de componentes para 95% varianza
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# k = ejercicio_1()
# print(f"\nComponentes para 95% varianza: {k}")

### 🟡 Ejercicio 2: PCA para Detección de Anomalías

Usa el error de reconstrucción para detectar outliers.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Detectar anomalías usando PCA
    
    Instrucciones:
    1. Genera datos normales + algunos outliers
       Ej: make_classification + agregar puntos extremos
    2. Aplica PCA reteniendo k < d componentes
    3. Para cada punto:
       a) Proyectar a espacio PCA
       b) Reconstruir a espacio original
       c) Calcular error de reconstrucción
    4. Puntos con error alto = anomalías
    5. Visualiza: datos normales vs anomalías detectadas
    
    Returns:
    --------
    metrics : dict
        Precision, recall para detección de anomalías
    """
    # TODO: Tu código aquí
    # Pista: threshold = mean + k*std del error de reconstrucción
    
    pass

# Descomentar para probar
# metrics = ejercicio_2()
# print("\n💡 PCA puede detectar outliers por su alto error de reconstrucción")

### 🔴 Ejercicio 3: Kernel PCA

Implementa una versión no-lineal de PCA usando kernel trick.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Implementar Kernel PCA
    
    Instrucciones:
    1. Genera datos con estructura no-lineal (ej: Swiss roll, círculos)
    2. Implementa Kernel PCA:
       a) Calcula matriz de kernel K (RBF)
       b) Centra K: K_centered = K - 1_n*K - K*1_n + 1_n*K*1_n
       c) Eigendecomposición de K_centered
       d) Proyectar: alpha = eigenvectors
    3. Compara visualizaciones:
       - PCA lineal (falla)
       - Kernel PCA (captura no-linealidad)
       - t-SNE (referencia)
    
    Returns:
    --------
    comparison : dict
        Visualizaciones y métricas
    """
    # TODO: Tu código aquí
    # Pista: sklearn.decomposition.KernelPCA para comparar
    
    pass

# Descomentar para probar
# comparison = ejercicio_3()
# print("\n🎯 Kernel PCA extiende PCA a relaciones no-lineales")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **PCA encuentra direcciones de máxima varianza**
   - Proyecta datos a espacio de menor dimensión
   - Minimiza pérdida de información
   - Componentes principales son ortogonales

2. **Matemática: Eigenvalues y Eigenvectors**
   - Componentes principales = eigenvectors de matriz de covarianza
   - Eigenvalues = varianza en cada dirección
   - SVD es método más estable numéricamente

3. **Varianza explicada**
   - Cuánta información captura cada componente
   - Regla práctica: retener 95% de varianza
   - Scree plot ayuda a elegir k

4. **Preprocessing esencial:**
   - **Siempre centrar** datos (restar media)
   - **Normalizar** si features en diferentes escalas
   - Considerar eliminar outliers primero

5. **Aplicaciones principales:**
   - Visualización (proyección a 2D/3D)
   - Compresión de datos
   - Preprocessing para ML
   - Feature extraction
   - Detección de anomalías

6. **Ventajas:**
   - ✅ Reduce overfitting
   - ✅ Acelera entrenamiento
   - ✅ Decorrelaciona features
   - ✅ Eficiente computacionalmente
   - ✅ Base teórica sólida

7. **Limitaciones:**
   - ❌ Solo captura relaciones lineales
   - ❌ Pierde interpretabilidad
   - ❌ Sensible a outliers
   - ❌ Sensible a escala (requiere normalización)
   - ❌ Ignora labels (unsupervised)

8. **Alternativas:**
   - **LDA:** Supervised, maximiza separabilidad de clases
   - **t-SNE:** No-lineal, mejor para visualización
   - **UMAP:** No-lineal, preserva estructura global y local
   - **Kernel PCA:** PCA no-lineal con kernel trick
   - **Autoencoder:** Aprendizaje profundo para reducción

---

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"On Lines and Planes of Closest Fit"** - Karl Pearson (1901)
  - Paper original de PCA
  - Philosophical Magazine

- **"Analysis of a Complex of Statistical Variables into Principal Components"** - Hotelling (1933)
  - Formalización matemática moderna

- **"Eigenfaces for Recognition"** - Turk & Pentland (1991)
  - Aplicación famosa de PCA a visión computacional
  - Journal of Cognitive Neuroscience

- **"Kernel PCA"** - Schölkopf et al. (1998)
  - Extensión no-lineal de PCA
  - Neural Computation

#### 📖 Libros Recomendados

- **"The Elements of Statistical Learning"** - Hastie et al.
  - Capítulo 14.5: Principal Components
  - Tratamiento matemático riguroso

- **"Pattern Recognition and Machine Learning"** - Bishop
  - Capítulo 12: Continuous Latent Variables
  - Conexión con modelos probabilísticos

- **"Introduction to Linear Algebra"** - Gilbert Strang
  - Para entender eigenvalues/eigenvectors profundamente

#### 🎥 Videos Recomendados

- **3Blue1Brown: Eigenvalues and Eigenvectors**
  - Visualización geométrica excelente
  - https://www.youtube.com/watch?v=PFDu9oVAE-g

- **StatQuest: PCA Step-by-Step**
  - Josh Starmer, muy claro
  - https://www.youtube.com/watch?v=FgakZw6K1QQ

- **Stanford CS229: PCA** - Andrew Ng
  - Tratamiento matemático completo

#### 💻 Documentación y Tutoriales

- [Scikit-learn: PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca)
- [PCA Visually Explained](https://setosa.io/ev/principal-component-analysis/)
- [Understanding PCA](https://towardsdatascience.com/a-one-stop-shop-for-principal-component-analysis-5582fb7e0a9c)

---

### 🤔 Preguntas para Reflexionar

1. **¿Por qué PCA requiere centrar pero no necesariamente escalar?**
   - Pista: Covarianza vs correlación

2. **¿Puede PCA aumentar el accuracy de un clasificador?**
   - Considera regularización vs pérdida de información

3. **¿Qué pasa si aplicamos PCA a datos ya decorrelacionados?**
   - Piensa en matriz de covarianza diagonal

4. **¿PCA es único?**
   - ¿Qué pasa con eigenvalues repetidos?

5. **¿Cómo se relaciona PCA con SVD?**
   - Conexión entre eigenvalues y valores singulares

---

## 🎓 Conclusión del Curso ML Clásico

**¡Felicidades!** Has completado la ruta de **Machine Learning Clásico**.

### Lo que aprendiste:

1. **Supervisado - Regresión:** Regresión Lineal, Gradient Descent
2. **Supervisado - Clasificación:** Logística, Softmax, Árboles, Random Forests, Boosting, SVM
3. **No Supervisado - Clustering:** K-Means
4. **No Supervisado - Dimensionalidad:** PCA

### Habilidades adquiridas:

- ✅ Implementar algoritmos desde cero
- ✅ Entender la matemática subyacente
- ✅ Aplicar con frameworks (Scikit-learn, XGBoost, etc.)
- ✅ Visualizar y interpretar resultados
- ✅ Elegir el algoritmo apropiado para cada problema

### Próximos pasos:

- **Ruta 2: RL Clásico** - Aprendizaje por refuerzo
- **Ruta 3: LLM Agents** - Agentes basados en modelos de lenguaje
- **Proyectos:** Aplicar lo aprendido a problemas reales

---

<div align="center">

**🎉 ¡Has dominado ML Clásico! 🎉**

**Continúa tu viaje:** [Ruta 02: RL Clásico](../02-rl-clasico/README.md)

[← 09. K-Means](09-kmeans.ipynb) | [Volver al índice](README.md)

</div>